<a href="https://colab.research.google.com/github/ShaoEH/ai-hw-spring-2026-nn/blob/main/mnist_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [2]:
mnist_mean = (0.1307,)
mnist_std = (0.3081,)

trainTransform = transforms.Compose([
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mnist_mean, mnist_std)
])

testTransform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mnist_mean, mnist_std)
])

In [3]:
trainset = torchvision.datasets.MNIST(root='./data', train=True, transform=trainTransform, download=True)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testset = torchvision.datasets.MNIST(root='./data', train=False, transform=testTransform, download=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.70MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.61MB/s]


In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, 3, 1)
        self.conv2 = nn.Conv2d(64, 128, 3, 1)
        self.dropout1 = nn.Dropout(0.3)
        self.fc1 = nn.Linear(18432, 256)
        self.dropout2 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return x

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

In [6]:
def train(total_rounds=5):
    model.train()
    history = []
    for round_idx in range(total_rounds):
        total_err = 0.0
        for i, (imgs, lbls) in enumerate(trainloader):
            imgs, lbls = imgs.to(device), lbls.to(device)

            optimizer.zero_grad()
            preds = model(imgs)
            batch_loss = criterion(preds, lbls)

            batch_loss.backward()
            optimizer.step()

            total_err += batch_loss.item()

        avg_err = total_err / len(trainloader)
        history.append(avg_err)
        print(f"Round {round_idx + 1} | Avg Loss: {avg_err:.5f}")
    return history

In [7]:
def test(data_loader):
    model.eval()
    correct_count = 0

    with torch.no_grad():
        for imgs, lbls in data_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)

            scores = model(imgs)
            preds = scores.argmax(dim=1, keepdim=True)

            correct_count += preds.eq(lbls.view_as(preds)).sum().item()

    final_score = 100. * correct_count / len(data_loader.dataset)
    print(f'\n[Validation] Accuracy: {final_score:.2f}%')

    return final_score

In [ ]:
losses = train(total_rounds=5)
final_acc = test(testloader)

In [ ]:
plt.plot(losses)
plt.title(f'Training Loss (Final Accuracy: {final_acc:.2f}%)')
plt.xlabel('Round')
plt.ylabel('Loss')
plt.savefig('test_results.png')
plt.show()